# Credit Card Fraud Detection — Data Cleaning

**Project:** Card Fraud Triage  
**Notebook:** Data quality assessment and cleaning  
**Dataset:** Credit Card Fraud Detection (`creditcard.csv`)

## Objective

This notebook inspects the raw transaction dataset, documents data-quality issues, applies a small set of justified cleaning steps, validates the result, and saves the cleaned dataset with an audit report.

The goal is to prepare reliable data for the next stages of the project without removing potentially useful fraud signals.

### Cleaning policy

- Validate expected columns, data types, and target labels (raise validation error on invalid labels).
- Inspect missing values, infinite values, exact duplicates, and basic transaction fields.
- Convert positive/negative infinity to missing values, then remove rows that remain incomplete.
- Remove exact duplicate rows and record how many removed rows were labelled as fraud.
- Retain zero-amount transactions and valid statistical outliers.
- Do **not** scale features or perform train/test splitting in this notebook; those steps belong in preprocessing/modeling.
- Keep `Class` for evaluation and for selecting legitimate training examples, but never use it as a GMM input feature.

> **Reproducibility note:** Run the notebook from the project root or from the `notebooks/` directory. By default, it imports paths and seed from `src/config.py`, expects `data/raw/creditcard.csv`, and writes the cleaned file to `data/processed/creditcard_clean.csv`.


## 1. Imports and project paths

Import standard libraries, configure reproducible random seeds, and import project paths from `src/config.py` to maintain a single source of truth across the repository.


In [1]:
import json
import os
import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# Find the project root from either the repository root or notebooks/.
cwd = Path.cwd().resolve()
possible_roots = [cwd, cwd.parent]
PROJECT_ROOT = next(
    (path for path in possible_roots if (path / "src" / "config.py").is_file()),
    cwd
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ARTIFACTS, CLEAN_PATH, RAW_PATH, SEED

# Set configured random seed for reproducibility
random.seed(SEED)
np.random.seed(SEED)

RAW_DATA_PATH = PROJECT_ROOT / RAW_PATH
CLEAN_DATA_PATH = PROJECT_ROOT / CLEAN_PATH
REPORT_PATH = PROJECT_ROOT / ARTIFACTS / "clean_report.json"

print(f"Project root:       {PROJECT_ROOT}")
print(f"Configured seed:    {SEED}")
print(f"Raw dataset path:   {RAW_DATA_PATH}")
print(f"Clean dataset path: {CLEAN_DATA_PATH}")
print(f"Report path:        {REPORT_PATH}")


Project root:       /Users/yuvrajchandravansi/Desktop/card-fraud-triage
Configured seed:    42
Raw dataset path:   /Users/yuvrajchandravansi/Desktop/card-fraud-triage/data/raw/creditcard.csv
Clean dataset path: /Users/yuvrajchandravansi/Desktop/card-fraud-triage/data/processed/creditcard_clean.csv
Report path:        /Users/yuvrajchandravansi/Desktop/card-fraud-triage/artifacts/clean_report.json


## 2. Load the raw dataset

The raw dataset is read without changing its contents. The original DataFrame is kept as `df_raw` so that before-and-after comparisons can be made later.


In [2]:
if not RAW_DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find the raw dataset at: {RAW_DATA_PATH}\n"
        "Place creditcard.csv in data/raw/ or update RAW_DATA_PATH above."
    )

df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Raw dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
display(df_raw.head())


Raw dataset shape: 284,807 rows × 31 columns


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0000,-1.3598,-0.0728,2.5363,1.3782,-0.3383,0.4624,0.2396,0.0987,0.3638,0.0908,-0.5516,-0.6178,-0.9914,-0.3112,1.4682,-0.4704,0.2080,0.0258,0.4040,0.2514,-0.0183,0.2778,-0.1105,0.0669,0.1285,-0.1891,0.1336,-0.0211,149.6200,0
1,0.0000,1.1919,0.2662,0.1665,0.4482,0.0600,-0.0824,-0.0788,0.0851,-0.2554,-0.1670,1.6127,1.0652,0.4891,-0.1438,0.6356,0.4639,-0.1148,-0.1834,-0.1458,-0.0691,-0.2258,-0.6387,0.1013,-0.3398,0.1672,0.1259,-0.0090,0.0147,2.6900,0
2,1.0000,-1.3584,-1.3402,1.7732,0.3798,-0.5032,1.8005,0.7915,0.2477,-1.5147,0.2076,0.6245,0.0661,0.7173,-0.1659,2.3459,-2.8901,1.1100,-0.1214,-2.2619,0.5250,0.2480,0.7717,0.9094,-0.6893,-0.3276,-0.1391,-0.0554,-0.0598,378.6600,0
3,1.0000,-0.9663,-0.1852,1.7930,-0.8633,-0.0103,1.2472,0.2376,0.3774,-1.3870,-0.0550,-0.2265,0.1782,0.5078,-0.2879,-0.6314,-1.0596,-0.6841,1.9658,-1.2326,-0.2080,-0.1083,0.0053,-0.1903,-1.1756,0.6474,-0.2219,0.0627,0.0615,123.5000,0
4,2.0000,-1.1582,0.8777,1.5487,0.4030,-0.4072,0.0959,0.5929,-0.2705,0.8177,0.7531,-0.8228,0.5382,1.3459,-1.1197,0.1751,-0.4514,-0.2370,-0.0382,0.8035,0.4085,-0.0094,0.7983,-0.1375,0.1413,-0.2060,0.5023,0.2194,0.2152,69.9900,0


### Inspect raw dataset information and memory usage

Review column names, non-null counts, inferred data types, and total memory footprint before performing cleaning operations.


In [3]:
print("Dataset information:")
df_raw.info()


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  fl

## 3. Validate the schema and target labels

The standard transaction dataset contains `Time`, `V1`–`V28`, `Amount`, and `Class`. This section validates column presence, uniqueness, numeric types, non-negative amounts, and allowed `Class` labels (`{0, 1}`). Invalid labels or missing columns raise an immediate `ValueError`.


In [4]:
expected_columns = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]

missing_columns = [column for column in expected_columns if column not in df_raw.columns]
extra_columns = [column for column in df_raw.columns if column not in expected_columns]
non_numeric_columns = [
    column for column in df_raw.columns
    if not pd.api.types.is_numeric_dtype(df_raw[column])
]
duplicate_column_count = int(df_raw.columns.duplicated().sum())

invalid_classes = []
if "Class" in df_raw.columns:
    invalid_classes = sorted(set(df_raw["Class"].dropna().unique()) - {0, 1})

negative_amounts = 0
if "Amount" in df_raw.columns:
    negative_amounts = int((df_raw["Amount"].dropna() < 0).sum())

schema_report = pd.DataFrame(
    {
        "Check": [
            "Required columns missing",
            "Unexpected extra columns",
            "Non-numeric columns",
            "Duplicate column names",
            "Invalid Class labels (outside {0, 1})",
            "Negative Amount values",
        ],
        "Result": [
            ", ".join(missing_columns) if missing_columns else "None",
            ", ".join(extra_columns) if extra_columns else "None",
            ", ".join(non_numeric_columns) if non_numeric_columns else "None",
            duplicate_column_count,
            ", ".join(map(str, invalid_classes)) if invalid_classes else "None",
            negative_amounts,
        ],
    }
)
display(schema_report)

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")
if duplicate_column_count:
    raise ValueError("Duplicate column names were found. Resolve them before cleaning.")
if non_numeric_columns:
    raise ValueError(f"Expected numeric data, but found non-numeric columns: {non_numeric_columns}")
if invalid_classes:
    raise ValueError(f"Class contains invalid values outside {{0, 1}}: {invalid_classes}")
if negative_amounts > 0:
    raise ValueError(f"Amount contains {negative_amounts} negative values.")

print("Schema and target label validation passed.")


,Check,Result
0,Required columns missing,None
1,Unexpected extra columns,None
2,Non-numeric columns,None
3,Duplicate column names,0
4,"Invalid Class labels (outside {0, 1})",None
5,Negative Amount values,0


Schema and target label validation passed.


## 4. Inspect data quality before cleaning

Measure missing values, infinite numeric values, exact duplicate rows, and basic label/amount issues to establish a pre-cleaning baseline.


In [5]:
numeric_raw = df_raw.select_dtypes(include=np.number)
infinite_mask = np.isinf(numeric_raw.to_numpy())

quality_before = pd.Series(
    {
        "Rows": len(df_raw),
        "Columns": len(df_raw.columns),
        "Missing cells": int(df_raw.isna().sum().sum()),
        "Rows with missing values": int(df_raw.isna().any(axis=1).sum()),
        "Infinite numeric cells": int(infinite_mask.sum()),
        "Rows with infinite values": int(infinite_mask.any(axis=1).sum()),
        "Exact duplicate rows": int(df_raw.duplicated().sum()),
        "Rows with invalid/missing Class": int((~df_raw["Class"].isin([0, 1])).sum()),
        "Negative Amount values": int((df_raw["Amount"] < 0).sum()),
        "Zero-amount transactions": int((df_raw["Amount"] == 0).sum()),
        "Constant columns": int((df_raw.nunique(dropna=False) <= 1).sum()),
    },
    name="Before cleaning",
)
display(quality_before.to_frame())


,Before cleaning
Rows,284807
Columns,31
Missing cells,0
Rows with missing values,0
Infinite numeric cells,0
Rows with infinite values,0
Exact duplicate rows,1081
Rows with invalid/missing Class,0
Negative Amount values,0
Zero-amount transactions,1825


### Class distribution before cleaning

Inspect the baseline distribution of legitimate (`Class 0`) and fraud (`Class 1`) transactions and record the initial fraud percentage.


In [6]:
print("Class distribution before cleaning:")
class_counts_before = (
    df_raw["Class"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Class")
    .to_frame("Transaction count")
)
display(class_counts_before)

if df_raw["Class"].notna().any():
    class_rate = df_raw["Class"].value_counts(normalize=True, dropna=True).sort_index() * 100
    print(f"Fraud rate among non-missing labels: {class_rate.get(1, 0):.4f}%")


Class distribution before cleaning:


,Transaction count
Class,
0,284315
1,492


Fraud rate among non-missing labels: 0.1727%


## 5. Review duplicate rows and label conflicts

Exact duplicate rows are identical across every column, including `Class`. This section breaks down duplicates by label (fraud vs. legitimate) and verifies whether identical feature sets occur with conflicting labels.


In [7]:
exact_duplicate_mask = df_raw.duplicated(keep="first")
duplicate_rows = df_raw.loc[exact_duplicate_mask].copy()

print(f"Exact duplicate rows identified: {len(duplicate_rows):,}")
print(f"Duplicate rows labelled fraud: {int((duplicate_rows['Class'] == 1).sum()):,}")
print(f"Duplicate rows labelled legitimate: {int((duplicate_rows['Class'] == 0).sum()):,}")

feature_columns = [column for column in df_raw.columns if column != "Class"]
same_feature_rows = df_raw.loc[df_raw.duplicated(subset=feature_columns, keep=False)]

if same_feature_rows.empty:
    conflicting_label_groups = 0
else:
    labels_per_feature_group = (
        same_feature_rows.groupby(feature_columns, dropna=False)["Class"]
        .nunique(dropna=False)
    )
    conflicting_label_groups = int((labels_per_feature_group > 1).sum())

print(
    "Groups with identical feature values but conflicting labels:",
    conflicting_label_groups,
)


Exact duplicate rows identified: 1,081
Duplicate rows labelled fraud: 19
Duplicate rows labelled legitimate: 1,062


Groups with identical feature values but conflicting labels: 0


## 6. Apply the cleaning steps

The cleaning order is intentional:
1. Work on a copy of the raw dataset.
2. Replace `+inf` and `-inf` in numeric columns with `NaN`.
3. Remove rows containing missing values.
4. Review exact duplicate rows and their class distribution before removal.
5. Remove exact duplicate rows (`keep="first"`).
6. Preserve original column order and sort by `Time` using a stable sort (`kind="stable"`) to maintain original transaction arrival order.
7. Cast `Class` to `int8` for memory efficiency.


In [8]:
df_clean = df_raw.copy()

# Step 1: Treat infinite numeric values as invalid/missing values.
numeric_columns = df_clean.select_dtypes(include=np.number).columns
df_clean[numeric_columns] = df_clean[numeric_columns].replace(
    [np.inf, -np.inf], np.nan
)

# Step 2: Remove rows with missing values, including values converted from infinity.
rows_before_missing_removal = len(df_clean)
df_clean = df_clean.dropna().copy()
rows_removed_for_missing_or_infinite = (
    rows_before_missing_removal - len(df_clean)
)

# Step 3: Review exact duplicate rows BEFORE removing them.
duplicate_mask = df_clean.duplicated(keep=False)
duplicate_review = df_clean.loc[duplicate_mask].copy()

print(f"Rows belonging to duplicate groups: {len(duplicate_review):,}")

print("\nClass distribution among duplicate rows:")
display(duplicate_review["Class"].value_counts().sort_index())

# Count only the extra copies that drop_duplicates will remove.
duplicate_rows_removed_preview = df_clean.duplicated(keep="first").sum()
print(
    "\nExtra duplicate rows that will be removed:",
    f"{duplicate_rows_removed_preview:,}"
)

# Step 4: Remove exact duplicate rows.
rows_before_deduplication = len(df_clean)
df_clean = df_clean.drop_duplicates(keep="first").copy()
duplicate_rows_removed = rows_before_deduplication - len(df_clean)

# Step 5: Restore column order and sort transactions by time using a stable sort.
df_clean = (
    df_clean.loc[:, df_raw.columns]
    .sort_values("Time", kind="stable")
    .reset_index(drop=True)
)
df_clean["Class"] = df_clean["Class"].astype("int8")

print(f"\nRows removed for missing/infinite values: {rows_removed_for_missing_or_infinite:,}")
print(f"Exact duplicate rows removed:             {duplicate_rows_removed:,}")
print(f"Clean dataset shape:                      {df_clean.shape[0]:,} × {df_clean.shape[1]}")


Rows belonging to duplicate groups: 1,854

Class distribution among duplicate rows:


Class
0    1822
1      32
Name: count, dtype: int64


Extra duplicate rows that will be removed: 1,081



Rows removed for missing/infinite values: 0
Exact duplicate rows removed:             1,081
Clean dataset shape:                      283,726 × 31


## 7. Compare the dataset before and after cleaning

The comparison verifies the exact effect of every cleaning step on row counts, duplicate count, nulls, and class balance.


In [9]:
def dataset_summary(frame):
    fraud_count = int((frame["Class"] == 1).sum())
    legitimate_count = int((frame["Class"] == 0).sum())
    total = len(frame)

    return {
        "Rows": total,
        "Columns": len(frame.columns),
        "Legitimate transactions": legitimate_count,
        "Fraud transactions": fraud_count,
        "Fraud rate (%)": (fraud_count / total * 100) if total else 0.0,
        "Missing cells": int(frame.isna().sum().sum()),
        "Exact duplicate rows": int(frame.duplicated().sum()),
    }

comparison = pd.DataFrame(
    [dataset_summary(df_raw), dataset_summary(df_clean)],
    index=["Before cleaning", "After cleaning"],
)
display(comparison)


,Rows,Columns,Legitimate transactions,Fraud transactions,Fraud rate (%),Missing cells,Exact duplicate rows
Before cleaning,284807,31,284315,492,0.1727,0,1081
After cleaning,283726,31,283253,473,0.1667,0,0


### Class distribution after cleaning

Verify the final class counts and total rows removed in the cleaned dataset.


In [10]:
print("Class distribution after cleaning:")
display(
    df_clean["Class"]
    .value_counts()
    .sort_index()
    .rename_axis("Class")
    .to_frame("Transaction count")
)

print(f"Rows removed in total: {len(df_raw) - len(df_clean):,}")


Class distribution after cleaning:


,Transaction count
Class,
0,283253
1,473


Rows removed in total: 1,081


## 8. Inspect transaction amounts and zero-amount rows

A zero transaction amount is not automatically an error. In financial fraud triage, zero-amount transactions frequently correspond to pre-authorization pings or card-testing behavior, and are therefore explicitly retained.


In [11]:
amount_summary = (
    df_clean.groupby("Class")["Amount"]
    .agg(["count", "mean", "median", "min", "max", "std"])
    .rename(index={0: "Legitimate (Class 0)", 1: "Fraud (Class 1)"})
)
display(amount_summary)

zero_amount_by_class = (
    df_clean.loc[df_clean["Amount"] == 0, "Class"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename(index={0: "Legitimate (Class 0)", 1: "Fraud (Class 1)"})
    .to_frame("Zero-amount transactions")
)
display(zero_amount_by_class)
print("Zero-amount transactions are retained.")


,count,mean,median,min,max,std
Class,,,,,,
Legitimate (Class 0),283253,88.4136,22.0000,0.0000,"25,691.1600",250.3790
Fraud (Class 1),473,123.8719,9.8200,0.0000,"2,125.8700",260.2110


,Zero-amount transactions
Class,
Legitimate (Class 0),1783
Fraud (Class 1),25


Zero-amount transactions are retained.


## 9. Flag potential outliers for review — do not remove them automatically

The interquartile range (IQR) rule is used as a diagnostic summary only. Features such as `V1`–`V28` are transformed variables, and unusual values may be essential indicators of fraudulent activity. Therefore, IQR-flagged observations are not deleted or clipped.


In [12]:
model_feature_columns = [column for column in df_clean.columns if column != "Class"]
features_for_review = df_clean[model_feature_columns]

q1 = features_for_review.quantile(0.25)
q3 = features_for_review.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_counts = (
    features_for_review.lt(lower_bound, axis="columns").sum()
    + features_for_review.gt(upper_bound, axis="columns").sum()
)

outlier_report = pd.DataFrame(
    {
        "IQR-flagged rows": outlier_counts.astype(int),
        "IQR-flagged percentage": (
            outlier_counts / len(features_for_review) * 100
        ).round(3),
    }
).sort_values("IQR-flagged percentage", ascending=False)

display(outlier_report)
print("These counts are diagnostic; no IQR-flagged rows were removed.")


,IQR-flagged rows,IQR-flagged percentage
V27,38799,13.6750
Amount,31685,11.1670
V28,30094,10.6070
V20,27553,9.7110
V8,23904,8.4250
V6,22886,8.0660
V23,18467,6.5090
V12,15282,5.3860
V21,14401,5.0760
V14,14060,4.9550


These counts are diagnostic; no IQR-flagged rows were removed.


## 10. Final validation checks

These assertions confirm that the saved dataset has no missing/infinite values or exact duplicates, that its labels are valid, and that its original columns are preserved.


In [13]:
assert len(df_clean) > 0, "The cleaned dataset is empty."
assert list(df_clean.columns) == list(df_raw.columns), "Column names/order changed unexpectedly."
assert not df_clean.columns.duplicated().any(), "Duplicate column names remain."
assert not df_clean.isna().any().any(), "Missing values remain."
assert not np.isinf(df_clean.select_dtypes(include=np.number).to_numpy()).any(), "Infinite values remain."
assert not df_clean.duplicated().any(), "Exact duplicate rows remain."
assert df_clean["Class"].isin([0, 1]).all(), "Invalid target labels remain."
assert df_clean["Time"].is_monotonic_increasing, "Rows are not sorted by Time."
assert (df_clean["Amount"] >= 0).all(), "Negative Amount values remain."

print("All final validation checks passed.")


All final validation checks passed.


## 11. Save the cleaned dataset and audit report

The cleaned CSV is saved without a pandas index. The JSON report records dataset sizes, class counts, and row removal counts for full auditability and reproducibility.


In [14]:
CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(CLEAN_DATA_PATH, index=False)

report = {
    "project": "Card Fraud Triage",
    "raw_dataset": str(RAW_DATA_PATH),
    "clean_dataset": str(CLEAN_DATA_PATH),
    "raw_shape": [int(df_raw.shape[0]), int(df_raw.shape[1])],
    "clean_shape": [int(df_clean.shape[0]), int(df_clean.shape[1])],
    "rows_removed": {
        "missing_or_infinite_values": int(rows_removed_for_missing_or_infinite),
        "exact_duplicates": int(duplicate_rows_removed),
        "total": int(len(df_raw) - len(df_clean)),
    },
    "removed": {
        "missing_values": int(quality_before["Missing cells"]),
        "rows_with_missing_values": int(rows_removed_for_missing_or_infinite),
        "duplicate_rows": int(duplicate_rows_removed),
    },
    "duplicate_analysis": {
        "duplicate_rows_labelled_fraud": int((duplicate_rows["Class"] == 1).sum()),
        "duplicate_rows_labelled_legitimate": int((duplicate_rows["Class"] == 0).sum()),
        "identical_feature_groups_with_conflicting_labels": int(conflicting_label_groups),
    },
    "before_cleaning": dataset_summary(df_raw),
    "after_cleaning": dataset_summary(df_clean),
    "before": {
        "rows": int(len(df_raw)),
        "duplicates": int(df_raw.duplicated().sum()),
        "nulls": int(df_raw.isna().sum().sum()),
        "frauds": int((df_raw["Class"] == 1).sum()),
        "fraud_rate": float((df_raw["Class"] == 1).sum() / len(df_raw)) if len(df_raw) else 0.0,
    },
    "after": {
        "rows": int(len(df_clean)),
        "duplicates": int(df_clean.duplicated().sum()),
        "nulls": int(df_clean.isna().sum().sum()),
        "frauds": int((df_clean["Class"] == 1).sum()),
        "fraud_rate": float((df_clean["Class"] == 1).sum() / len(df_clean)) if len(df_clean) else 0.0,
    },
    "zero_amount_rows_retained": int((df_clean["Amount"] == 0).sum()),
    "outlier_policy": "IQR flags are diagnostic only; no outliers were removed.",
    "preprocessing_policy": (
        "Scaling and train/validation/test splitting are intentionally deferred "
        "to a separate preprocessing notebook."
    ),
    "modeling_note": (
        "Exclude Class from GMM input features. Use Class only to select legitimate "
        "training rows and evaluate anomaly detection."
    ),
}

REPORT_PATH.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")

print(f"Cleaned dataset saved to: {CLEAN_DATA_PATH}")
print(f"Audit report saved to:    {REPORT_PATH}")


Cleaned dataset saved to: /Users/yuvrajchandravansi/Desktop/card-fraud-triage/data/processed/creditcard_clean.csv
Audit report saved to:    /Users/yuvrajchandravansi/Desktop/card-fraud-triage/artifacts/clean_report.json


## 12. Conclusion and handoff to the next stage

The cleaned dataset is now saved in `data/processed/creditcard_clean.csv` and ready for exploratory data analysis and preprocessing.

### What this notebook did
- Validated expected columns, data types, and confirmed valid `{0, 1}` target labels.
- Handled potential infinite values and removed exact duplicate records.
- Audited dropped duplicate records by class (19 fraud, 1,062 legitimate).
- Confirmed zero label conflicts across identical feature vectors.
- Retained zero-amount transactions (1,808 rows) and documented outlier diagnostics.
- Saved the cleaned dataset to `data/processed/creditcard_clean.csv` and audit report to `artifacts/clean_report.json`.

**Next step:** perform EDA on `data/processed/creditcard_clean.csv`, then define a leakage-safe data split in preprocessing.
